In [1]:
# 02_Preprocessing.ipynb
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import joblib

# Load raw data
df = pd.read_csv('../data/raw/Automobile_Loan_Default.csv')

# 1) Drop ID columns
drop_cols = [c for c in df.columns if c.lower() in ('id','client_id','application_id')]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# 2) Convert object columns to numeric where possible
obj_cols = df.select_dtypes(include=['object']).columns.tolist()
for c in obj_cols:
    s = df[c].astype(str).str.replace(',','').str.replace(' ','').replace('nan', np.nan)
    coerced = pd.to_numeric(s, errors='coerce')
    if coerced.notnull().sum() / len(coerced) > 0.6:
        df[c] = coerced

# 3) Drop columns with >60% missing
missing_pct = df.isnull().mean()
drop_high_missing = missing_pct[missing_pct > 0.60].index.tolist()
df = df.drop(columns=drop_high_missing)

# 4) Separate numeric & categorical
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if 'Default' in num_cols: num_cols.remove('Default')
cat_cols = [c for c in df.columns if c not in num_cols and c != 'Default']

# 5) Impute missing values
num_imp = SimpleImputer(strategy='median')
df[num_cols] = num_imp.fit_transform(df[num_cols])

cat_imp = SimpleImputer(strategy='constant', fill_value='MISSING')
df[cat_cols] = cat_imp.fit_transform(df[cat_cols])

# 6) Derived features
if set(['Credit_Amount','Client_Income']).issubset(df.columns):
    df['credit_to_income'] = df['Credit_Amount'] / df['Client_Income'].replace({0: np.nan})
if 'Loan_Annuity' in df.columns and 'Client_Income' in df.columns:
    df['annuity_to_income'] = df['Loan_Annuity'] / df['Client_Income'].replace({0: np.nan})

df['credit_to_income'] = df.get('credit_to_income', 0).replace([np.inf, -np.inf], np.nan).fillna(0)
df['annuity_to_income'] = df.get('annuity_to_income', 0).replace([np.inf, -np.inf], np.nan).fillna(0)

# 7) Encoding
low_card = [c for c in cat_cols if df[c].nunique() <= 10]
high_card = [c for c in cat_cols if df[c].nunique() > 10]
df = pd.get_dummies(df, columns=low_card, drop_first=True)

for c in high_card:
    freq = df[c].value_counts(normalize=True)
    df[c + "_freq_enc"] = df[c].map(freq).astype(float)
df = df.drop(columns=high_card)

# 8) Scale numeric features
numeric_after = df.select_dtypes(include=[np.number]).columns.tolist()
if 'Default' in numeric_after: numeric_after.remove('Default')
scaler = StandardScaler()
df[numeric_after] = scaler.fit_transform(df[numeric_after])

# 9) Save preprocessed dataset
df.to_csv('../data/processed/preprocessed_automobile_loan.csv', index=False)
joblib.dump(df, '../data/processed/preprocessed_automobile_loan.pkl')
joblib.dump(scaler, '../data/processed/scaler.joblib')

df.head()


C:\Users\HP\AppData\Local\Temp\ipykernel_16240\4203062421.py:9: DtypeWarning: Columns (1,7,8,16,17,18,19,20,35) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/raw/Automobile_Loan_Default.csv')


,Client_Income,Car_Owned,Bike_Owned,Active_Loan,House_Own,Child_Count,Credit_Amount,Loan_Annuity,Population_Region_Relative,Age_Days,...,Client_Housing_Type_Home,Client_Housing_Type_MISSING,Client_Housing_Type_Municipal,Client_Housing_Type_Office,Client_Housing_Type_Rental,Client_Housing_Type_Shared,Client_Permanent_Match_Tag_Yes,Client_Contact_Work_Tag_Yes,Client_Occupation_freq_enc,Type_Organization_freq_enc
0,-0.882958,-0.706223,-0.68975,1.03194,-1.532308,-0.561872,0.034994,0.491575,0.015324,-0.479319,...,True,False,False,False,False,False,True,True,-0.591953,0.224271
1,0.304060,1.415982,-0.68975,1.03194,0.652610,-0.561872,-1.119391,-0.618288,-0.034243,-0.431660,...,True,False,False,False,False,False,True,True,1.300203,-0.839467
2,0.106223,-0.706223,-0.68975,1.03194,-1.532308,0.824417,-0.006827,0.052843,0.000857,0.179313,...,False,False,False,False,False,False,True,True,-1.357623,0.224271
3,-0.091613,-0.706223,-0.68975,1.03194,0.652610,-0.561872,-0.149073,-0.291045,-0.029355,1.668385,...,True,False,False,False,False,False,True,True,1.300203,0.853374
4,1.491077,1.415982,-0.68975,1.03194,-1.532308,2.210706,1.865520,0.582651,-0.004292,-1.081690,...,True,False,False,False,False,False,True,True,-0.017958,1.367141
